In [8]:
import json
import re
import csv
from pathlib import Path
from collections import Counter
from math import log, sqrt

import numpy as np
from nltk.stem import PorterStemmer

# seed for reproducibility
SEED = 42
np.random.seed(SEED)

In [23]:
# define data directory and check for required files
DATA_DIR = Path("../../data").resolve()
dev_file = DATA_DIR / "dev_labeled_ChatGPT.jsonl"
passages_file = DATA_DIR / "passages_bm25_ChatGPT_humfacts.jsonl"

In [10]:
def read_jsonl(file_path):
    """Read a JSON Lines file into a list of dictionaries."""
    records = []

    with open(file_path, "r", encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if not line.strip():
                continue

            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(
                    f"Invalid JSON in {file_path}, line {line_number}"
                ) from exc

    return records

In [ ]:
# Load the development and passage data
dev_data = read_jsonl(dev_file)
passages_data = read_jsonl(passages_file)

# Display information about the loaded data
print(f"Data directory: {DATA_DIR}")
print(f"Loaded {len(dev_data)} development records.")
print(f"Loaded {len(passages_data)} passage records.")

Data directory: /Users/seyedrumaiz/Library/Mobile Documents/com~apple~CloudDocs/University/Year 3/CM3604/LLMFactChecking/data
Loaded 31 development records.
Loaded 5427 passage records.


In [ ]:
# Display keys and example records
print("Development record keys:")
if dev_data:
    print(list(dev_data[0].keys()))

print("\nPassage record keys:")
if passages_data:
    print(list(passages_data[0].keys()))

# Dsplay example records
if dev_data:
    print("\nExample development record:")
    print(json.dumps(dev_data[0], indent=2, ensure_ascii=False)[:2500])

# Display example passage record
if passages_data:
    print("\nExample passage record:")
    print(json.dumps(passages_data[0], indent=2, ensure_ascii=False)[:2500])

Development record keys:
['input', 'output', 'topic', 'cat', 'annotations']

Passage record keys:
['name', 'sent', 'passages']

Example development record:
{
  "input": "Question: Tell me a bio of Marianne McAndrew.",
  "output": "Marianne McAndrew is an American actress and singer, born on November 21, 1942, in Cleveland, Ohio. She began her acting career in the late 1960s, appearing in various television shows and films. Her notable film appearances include the role of Irene Molloy in the 1969 film adaptation of the musical \"Hello, Dolly!\", and as Princess Zareta in the 1976 film \"The Blue Bird\". She also had recurring roles on popular television shows such as \"The Love Boat\" and \"Fantasy Island\". In addition to her acting career, McAndrew was a trained singer and performed in various stage productions, including the Broadway production of \"The King and I\". She retired from acting in the 1990s and currently resides in California.",
  "topic": "Marianne McAndrew",
  "cat": [

In [ ]:
# Binary mapping used in the earlier results:
# S = supported
# NS = not supported
# IR = irrelevant (treated as unsupported for this experiment)

RAW_LABEL_MAP = {
    "S": 1,
    "NS": 0,
    "IR": 0,
}

fact_to_passage = {}

for record in passages_data:
    fact_text = record.get("sent")
    passages = record.get("passages") or []

    if isinstance(fact_text, str):
        fact_to_passage[fact_text] = passages

In [15]:
examples = []

missing_passage_facts = 0
missing_label_facts = 0
unrecognized_label_facts = 0
observed_labels = Counter()

for record in dev_data:
    annotations = record.get("annotations") or []

    if isinstance(annotations, dict):
        annotations = [annotations]

    for sentence_annotation in annotations:
        atomic_facts = (
            sentence_annotation.get("human-atomic-facts") or []
        )

        for fact in atomic_facts:
            fact_text = fact.get("text")
            raw_label = fact.get("label")

            if not fact_text or raw_label is None:
                missing_label_facts += 1
                continue

            label = str(raw_label).strip().upper()
            observed_labels[label] += 1

            if label not in RAW_LABEL_MAP:
                unrecognized_label_facts += 1
                continue

            if fact_text not in fact_to_passage:
                missing_passage_facts += 1
                continue

            passages = fact_to_passage[fact_text] or []

            examples.append({
                "fact": fact_text,
                "passages": passages,
                "label": label,
                "gold_binary": RAW_LABEL_MAP[label],
            })